In [11]:
import os
import pyreadr
import numpy as np
import pandas as pd
from pprint import pprint
from loguru import logger
from typing import Optional, Tuple


def format_group_column(
    meta_df: pd.DataFrame, 
    column_name: str,
    prefix: str = "Group-",
    special_map: Optional[dict] = None
) -> pd.DataFrame:
    """
    Dynamically format group labels with a prefix for any number of numeric 
    groups, while replacing specific keywords based on a mapping dictionary.
    """
    # Default specific replacements if none provided
    if special_map is None:
        special_map = {"QC": "Invalid"}
        
    # Return early if column is missing
    if column_name not in meta_df.columns:
        return meta_df
        
    # Cast to string to prevent integer vs string matching bugs
    meta_df[column_name] = meta_df[column_name].astype(str)
    
    def _apply_mapping(val: str) -> str:
        # 1. Handle explicit special cases first
        if val in special_map:
            return special_map[val]
        
        # 2. Dynamically add prefix to any numeric string 
        if val.isdigit():
            return f"{prefix}{val}"
            
        # 3. Fallback: return the original text if neither rule applies
        return val
        
    # Apply the mapping function element-wise
    meta_df[column_name] = meta_df[column_name].apply(_apply_mapping)
    
    return meta_df


def process_amide_dataset(
    file_path: str
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Parse raw RDa file, format it to pi-metaboqc standard requirements, 
    and extract aligned metadata and intensity DataFrames.
    """
    logger.info(f"Loading raw data from: {file_path}")
    parsed_data = pyreadr.read_r(file_path)
    
    # Log available objects instead of printing
    logger.info(f"Datasets found in RDa: {list(parsed_data.keys())}")
    
    # Extract the specific dataset and replace 0 with NaN
    int_df = parsed_data["Amide_data"].replace(0, np.nan)
    
    # Define original index levels and target standard columns
    original_idx = ["name", "injection.order", "class", "batch", "group"]
    standard_cols = [
        "Sample Name", "Inject Order", "Sample Type", "Batch", "Bio Group"
    ]
    
    logger.info("Transposing and standardizing index structures...")
    # Use parentheses for clean PEP 8 compliant method chaining
    int_df = (
        int_df.set_index(original_idx)
        .transpose()
        .rename_axis(columns=standard_cols, index=["Metabolite"])
    )
    
    # Extract metadata into a separate DataFrame
    meta = int_df.columns.to_frame().reset_index(drop=True)
    
    # Retain only the 'Sample Name' level for the intensity matrix columns
    int_df.columns = int_df.columns.get_level_values("Sample Name")
    
    logger.info("Formatting metadata content...")
    meta["Batch"] = "Batch-" + meta["Batch"].astype(str)
    meta["Sample Type"] = meta["Sample Type"].replace(
        "Subject", "Sample"
    )
    meta = format_group_column(meta_df=meta, column_name="Bio Group")
    
    # Sort metadata by injection sequence
    meta = meta.sort_values(by=["Inject Order"]).reset_index(drop=True)
    
    # Align intensity matrix columns strictly to the sorted metadata
    int_df = int_df[meta["Sample Name"]]
    
    logger.success("Data parsing and alignment completed successfully.")
    return meta, int_df

def summarize_batch_info(
    meta: pd.DataFrame,
    batch_col: str = "Batch",
    order_col: str = "Inject Order",
    type_col: str = "Sample Type"
) -> pd.DataFrame:
    """
    Summarize batch details including injection order range, sample counts, 
    QC counts, and the presence of any other sample categories.
    """
    # Verify that all required columns exist in the metadata
    missing = [
        c for c in [batch_col, order_col, type_col] if c not in meta.columns
    ]
    if missing:
        raise ValueError(f"Missing required columns in meta: {missing}")

    # Ensure injection order is numeric for accurate min/max calculations
    orders = pd.to_numeric(meta[order_col], errors="coerce")
    
    summary_records = []
    
    # Iterate through each analytical batch independently
    for batch_id, group in meta.groupby(batch_col):
        # 1. Determine the injection order range safely
        b_orders = orders[group.index]
        if b_orders.notna().any():
            order_range = f"{b_orders.min():.0f} ~ {b_orders.max():.0f}"
        else:
            order_range = "Unknown"
        
        # 2. Count the occurrences of each sample type
        type_counts = group[type_col].value_counts()
        sample_count = type_counts.get("Sample", 0)
        qc_count = type_counts.get("QC", 0)
        
        # Append the calculated metrics as a dictionary record
        summary_records.append({
            "Batch ID": batch_id,
            "Injection Order": order_range,
            "Samples": sample_count,
            "Pooled QCs": qc_count
        })
        
    # Convert the records list to a structured DataFrame
    summary_df = pd.DataFrame(summary_records)
    
    return summary_df

# if __name__ == "__main__":
# Define raw data path safely across different operating systems
input_dir = os.path.abspath(os.path.join(
    "..", "..", "data", "raw", "WaveICA"))
output_dir = os.path.abspath(os.path.join(
    "..", "..", "data", "processed", "WaveICA"))

os.makedirs(output_dir, exist_ok=True)
data_file = os.path.join(input_dir, "Amide_data.rda")

# Execute the preprocessing pipeline
metadata_df, intensity_df = process_amide_dataset(data_file)
pprint(summarize_batch_info(meta=metadata_df))

# Note: Output can now be saved to CSV files in the 'processed' folder
meta_path = os.path.join(output_dir,"project_meta.csv")
int_path = os.path.join(output_dir,"project_intensity.csv")

metadata_df.to_csv(
    path_or_buf = meta_path, index=None, header=True, 
    encoding="utf-8-sig")
logger.success(f"Meta data saved successfully in: {meta_path}.")

intensity_df.to_csv(
    path_or_buf = int_path,
    encoding="utf-8-sig", na_rep="N/A")
logger.success(f"Intensity data saved successfully in: {int_path}.")

2026-05-25 16:35:18.933 | INFO     | __main__:process_amide_dataset:56 - Loading raw data from: c:\Users\Complex\Desktop\Personal_repositories\pi-metaboqc-casestudy\data\raw\WaveICA\Amide_data.rda
2026-05-25 16:35:19.453 | INFO     | __main__:process_amide_dataset:60 - Datasets found in RDa: ['Amide_data']
2026-05-25 16:35:19.464 | INFO     | __main__:process_amide_dataset:71 - Transposing and standardizing index structures...
2026-05-25 16:35:19.470 | INFO     | __main__:process_amide_dataset:85 - Formatting metadata content...
2026-05-25 16:35:19.484 | SUCCESS  | __main__:process_amide_dataset:98 - Data parsing and alignment completed successfully.
2026-05-25 16:35:19.496 | SUCCESS  | __main__:<module>:171 - Meta data saved successfully in: c:\Users\Complex\Desktop\Personal_repositories\pi-metaboqc-casestudy\data\processed\WaveICA\project_meta.csv.


  Batch ID Injection Order  Samples  Pooled QCs
0  Batch-1         1 ~ 217      192          25
1  Batch-2       218 ~ 434      192          25
2  Batch-3       435 ~ 642      184          24
3  Batch-4       643 ~ 733       76          11


2026-05-25 16:35:21.661 | SUCCESS  | __main__:<module>:176 - Intensity data saved successfully in: c:\Users\Complex\Desktop\Personal_repositories\pi-metaboqc-casestudy\data\processed\WaveICA\project_intensity.csv.


In [12]:
pprint(metadata_df["Sample Type"].value_counts())
pprint(metadata_df.head(5))

Sample Type
Sample    644
QC         85
Name: count, dtype: int64
  Sample Name  Inject Order Sample Type    Batch Bio Group
0         QC1             1          QC  Batch-1   Invalid
1          A2             2      Sample  Batch-1   Group-1
2          A3             3      Sample  Batch-1   Group-1
3          A4             4      Sample  Batch-1   Group-1
4          A5             5      Sample  Batch-1   Group-1


In [10]:
pprint(intensity_df.shape)
pprint(intensity_df.head(5))

(6402, 729)
Sample Name            QC1             A2             A3             A4  \
Metabolite                                                                
Var1           7285.554663    8761.977029    8173.919338    7946.187898   
Var2           7899.774617    4750.776315    1067.502825   16063.522410   
Var3         301202.826500  355063.976000  247599.206100  311915.637400   
Var4          28466.379130   31496.106410   23622.386890   29951.228080   
Var5          18725.372180   24228.953950   29987.197290   24027.056240   

Sample Name            A5             A6             A7             A8  \
Metabolite                                                               
Var1           9806.94088   14876.139450    9466.228766    9213.458499   
Var2                  NaN    7906.974448            NaN    7084.360339   
Var3         291640.36100  381391.610300  306701.769700  266575.483100   
Var4          26986.95906   38652.499980   26950.448150   20606.180090   
Var5          2555